In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data, util

def load_image(path=''):
    if not path.strip():
        return data.camera()
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4), squeeze=False)
    for ax, im, title in zip(axes.ravel(), images, titles):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

img = load_image(input('Image path (Enter for sample): '))

In [ ]:
from scipy import stats
rng = np.random.default_rng(49)
shape = img.shape
def u():
    return np.clip(rng.random(shape),1e-12,1-1e-12)
a,b = -0.1,0.1
mu,sigma = 0,0.05
rate,k = 40,3
scale = 0.05
manual = {
    'Uniform': a+(b-a)*u(),
    'Gaussian': mu+sigma*np.sqrt(-2*np.log(u()))*np.cos(2*np.pi*u()),
    'Erlang': sum(-np.log(u())/rate for _ in range(k)),
    'Exponential': -np.log(u())/rate,
    'Rayleigh': scale*np.sqrt(-2*np.log(u()))
}
# Python has no MATLAB imnoise for these five modes; use equivalent library samplers.
builtin = {
    'Uniform': stats.uniform.rvs(loc=a,scale=b-a,size=shape,random_state=rng),
    'Gaussian': stats.norm.rvs(loc=mu,scale=sigma,size=shape,random_state=rng),
    'Erlang': stats.erlang.rvs(k,scale=1/rate,size=shape,random_state=rng),
    'Exponential': stats.expon.rvs(scale=1/rate,size=shape,random_state=rng),
    'Rayleigh': stats.rayleigh.rvs(scale=scale,size=shape,random_state=rng)
}
for name, noise in manual.items():
    other = builtin[name]
    noisy = np.clip(img/255.+noise,0,1)*255
    noisy_builtin = np.clip(img/255.+other,0,1)*255
    show([img,noisy,noisy_builtin],['Original',name+' manual',name+' library'])
    bins = np.linspace(min(noise.min(),other.min()),max(noise.max(),other.max()),80)
    plt.hist(noise.ravel(),bins=bins,density=True,alpha=.5,label='Manual')
    plt.hist(other.ravel(),bins=bins,density=True,alpha=.5,label='Library')
    plt.title(name+' noise distribution'); plt.legend(); plt.show()
# Direct comparison with Python's imnoise equivalent for Gaussian noise.
show([img,util.random_noise(img,mode='gaussian',mean=mu,var=sigma**2)*255],['Original','Gaussian random_noise'])